In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
# !wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/code/evaluation_utils.py

In [3]:
doc = documents[0]
print(doc["filename"])
print(doc["content"][:500])


01-agentic-rag/lessons/01-intro.md
# Introduction

Video: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In this module, we'll build a working Retrieval-Augmented
Generation (RAG) system from scratch, step by step.

We write everything in plain Python. We build a small search index by
hand and call the LLM ourselves. I want you to see every piece first.
That way you know what a framework does for you before you reach for
one.

Places where you can find me:

- [My substack


In [4]:
from dotenv import load_dotenv
load_dotenv()
from rag_helper import RAGBase
from evaluation_utils import RAGWithUsage
from google import genai

google_client = genai.Client()

In [5]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

# Q1. Generating questions
Generating questions for all 72 pages costs money and takes time, so let's start small and generate questions for just the first 3 pages:

01-agentic-rag/lessons/01-intro.md

01-agentic-rag/lessons/02-environment.md

01-agentic-rag/lessons/03-rag.md

Each call returns the token usage, which most LLM APIs report on the response object (e.g. response.usage.input_tokens / prompt_tokens).

What's the average number of input tokens across these 3 calls?

- 140
- 1400
- 14000
- 140000

In [6]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [7]:
import json
from tqdm.auto import tqdm

usages = []
questions_by_doc = []

for doc in documents[:3]:
    user_prompt = json.dumps(doc, indent=2)

    response = google_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=user_prompt,
        config={
            "system_instruction": data_gen_instructions,
            "response_mime_type": "application/json",
            "response_schema": Questions, # he shape of the JSON you want the model to return
        },
    )

    usages.append(response.usage_metadata)
    questions_by_doc.append(response.parsed)

In [8]:
print("Usages:", usages)

Usages: [GenerateContentResponseUsageMetadata(
  candidates_token_count=102,
  prompt_token_count=1070,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=1070
    ),
  ],
  thoughts_token_count=918,
  total_token_count=2090
), GenerateContentResponseUsageMetadata(
  candidates_token_count=143,
  prompt_token_count=1407,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=1407
    ),
  ],
  thoughts_token_count=967,
  total_token_count=2517
), GenerateContentResponseUsageMetadata(
  candidates_token_count=113,
  prompt_token_count=1893,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=1893
    ),
  ],
  thoughts_token_count=1263,
  total_token_count=3269
)]


In [9]:
# !wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/04-evaluation/ground-truth.csv

# Q2. First result with text search
Take the first question from the ground truth:

q = ground_truth[0]["question"]
After running text_search for it, what's the filename of the first result?

- 01-agentic-rag/lessons/01-intro.md
- 01-agentic-rag/lessons/03-rag.md
- 01-agentic-rag/lessons/13-function-calling.md
- 01-agentic-rag/lessons/10-rag-next-steps.md


In [10]:
# Searching the chunks

from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [11]:
# text search
from minsearch import Index


def text_search(query: str, num_results: int = 5):
    index = Index(text_fields=["content"])
    index.fit(documents)
    results = index.search(query)
    return [r["filename"] for r in results[:num_results]]
    


In [12]:
import pandas as pd

df_ground_truth = pd.read_csv("/workspaces/llm_hw/llm-zoomcamp-2026-hw/ground-truth.csv")
q = df_ground_truth.iloc[0]["question"]

text_results = text_search(q)
text_results

['01-agentic-rag/lessons/01-intro.md',
 '01-agentic-rag/lessons/03-rag.md',
 '01-agentic-rag/lessons/13-function-calling.md',
 '04-evaluation/lessons/01-intro.md',
 '04-evaluation/lessons/11-evaluation-intro.md']

In [13]:
print(df_ground_truth.head())

                                            question  \
0  What exactly is a retrieval-augmented generati...   
1  Why does this course build the RAG project in ...   
2  What are the main weaknesses of large language...   
3  What will the course build in the first part o...   
4  What kind of example app are you building here...   

                             filename  
0  01-agentic-rag/lessons/01-intro.md  
1  01-agentic-rag/lessons/01-intro.md  
2  01-agentic-rag/lessons/01-intro.md  
3  01-agentic-rag/lessons/01-intro.md  
4  01-agentic-rag/lessons/01-intro.md  


# Q3. First result with vector search
After running vector_search for the same question, what's the filename of the first result?

- 01-agentic-rag/lessons/01-intro.md
- 01-agentic-rag/lessons/03-rag.md
- 04-evaluation/lessons/11-evaluation-intro.md
- 04-evaluation/lessons/12-rag-answers.md

In [14]:
# vector search
from embedder import Embedder
embedder = Embedder()

from minsearch import VectorSearch
vsearch = VectorSearch()

import numpy as np


def vector_search(query,  num_results=5):
    document_vectors = [embedder.encode(doc["content"]) for doc in documents]
    document_vectors = np.array(document_vectors)
    vsearch.fit(document_vectors, documents)
    query_vector = embedder.encode(query)
    results = vsearch.search(query_vector, num_results=num_results)
    return [r["filename"] for r in results[:num_results]]



2026-07-18 03:27:43.063389642 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [15]:
vector_results = vector_search(q)
vector_results

['01-agentic-rag/lessons/01-intro.md',
 '04-evaluation/lessons/11-evaluation-intro.md',
 '01-agentic-rag/lessons/10-rag-next-steps.md',
 '06-best-practices/lessons/01-intro.md',
 '07-project-example/lessons/03-evaluating-rag.md']

# Q4. Evaluating text search
Evaluate text_search on the ground truth data.

What's the Hit Rate?

- 0.55
- 0.66
- 0.76
- 0.88

In [16]:
from tqdm.auto import tqdm

# Then turn this comparison into a relevance list. In this lesson, relevance means whether a retrieved document is the correct document for this question.


def compute_relevance(q, search_function):
    target_filename = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for r in results:
        relevance.append(int(r == target_filename))

    return relevance


def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for _, q in tqdm(ground_truth.iterrows(), total=len(ground_truth)):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total



relevance_total = compute_relevance_total(df_ground_truth, text_search)
relevance_total



def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

# Check it on the same example:

hit_rate(relevance_total)

  0%|          | 0/360 [00:00<?, ?it/s]

0.7555555555555555

# Q5. Evaluating vector search

Now evaluate vector_search - the part we left for the homework, since the module only evaluated keyword search.

What's the MRR?

- 0.35
- 0.45
- 0.55
- 0.65

In [17]:
relevance_total_vector = compute_relevance_total(df_ground_truth, vector_search)
relevance_total_vector

hit_rate(relevance_total_vector)

  0%|          | 0/360 [00:00<?, ?it/s]

0.6138888888888889

In [22]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)


In [23]:
mrr(relevance_total_vector)

0.4231018518518519

# Q6. Tuning hybrid search
The k constant in RRF controls how much the top ranks matter. A smaller k sharpens the gap between positions, so being at the top of a list counts for more. The RRF paper uses 60 as a default, but the best value depends on the data

so let's measure it.
Evaluate hybrid_search over the full ground truth dataset for k values 1, 50, 100, and 200. Compare the MRR values for these runs.

Which k gives the best MRR?

- 1
- 50
- 100
- 200

In [18]:
# hybrid search
def rrf(result_lists, k=60, num_results=5):
    scores = {}

    for results in result_lists:
        for rank, filename in enumerate(results):
            scores[filename] = scores.get(filename, 0) + 1 / (k + rank + 1)

    ranked = sorted(scores, key=scores.get, reverse=True)
    return ranked[:num_results]


# Then define hybrid_search on top of it:
def hybrid_search(query, k=60, num_results=5):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k, num_results=num_results)



In [ ]:
relevance_total_hybrid_1 = compute_relevance_total(
    df_ground_truth,
    lambda query: hybrid_search(query, k=1, num_results=5)
)


mrr(relevance_total_hybrid_1)


  0%|          | 0/360 [00:00<?, ?it/s]

In [ ]:
relevance_total_hybrid_50 = compute_relevance_total(
    df_ground_truth,
    lambda query: hybrid_search(query, k=50, num_results=5)
)


mrr(relevance_total_hybrid_50)


NameError: name 'mrr' is not defined

In [ ]:
relevance_total_hybrid_100 = compute_relevance_total(
    df_ground_truth,
    lambda query: hybrid_search(query, k=100, num_results=5)
)


mrr(relevance_total_hybrid_100)


In [ ]:
relevance_total_hybrid_200 = compute_relevance_total(
    df_ground_truth,
    lambda query: hybrid_search(query, k=200, num_results=5)
)
mrr(relevance_total_hybrid_200)